# SpatioCube end-to-end demo (MouseBrain)

这个 notebook 演示：**读取合并 h5ad → 按 `sampleid` 拆片 → 相邻切片 OT 对齐 → 3D 建图 + Leiden → 3D 可视化**。

## 环境依赖（建议 conda 环境）

最小（能跑到 Leiden 聚类）：

- Python >= 3.9
- `scanpy`, `anndata`, `numpy`, `scipy`, `pandas`, `scikit-learn`
- OT 对齐：`POT`（可选但本 notebook 默认会用）
- Leiden：`python-igraph`, `leidenalg`
- 可视化：`plotly`

可选：对比学习 embedding 需要 `torch`。


## 0) 配置数据路径

把你的合并 `.h5ad` 路径填到下面（或设置环境变量 `SPATIOCUBE_MOUSEBRAIN_H5AD`）。


In [1]:
import os

# 直接在这里指定
H5AD_PATH = os.environ.get(
    "SPATIOCUBE_MOUSEBRAIN_H5AD",
    "/cluster3/labData/jiamao/MouseBrain/h5ad/T300_T309_SCT_merge_bayes_anno.h5ad",
)
os.environ["SPATIOCUBE_MOUSEBRAIN_H5AD"] = H5AD_PATH
print("SPATIOCUBE_MOUSEBRAIN_H5AD =", os.environ["SPATIOCUBE_MOUSEBRAIN_H5AD"])

SPATIOCUBE_MOUSEBRAIN_H5AD = /cluster3/labData/jiamao/MouseBrain/h5ad/T300_T309_SCT_merge_bayes_anno.h5ad


## 1) 读取数据并拆片

- 默认按 `adata.obs['sampleid']` 拆片。
- 如果 `obsm['spatial']` 不存在，会从 `obs['coor_x_ad2'] / obs['coor_y_ad2']` 自动补。


In [2]:
import spatiocube as scb

adata = scb.read_merged_h5ad()  # uses env var
cube = scb.SpatioCube.from_merged_h5ad(
    adata,
    slice_key="sampleid",
    lambda_z=0.01,
    # XY：后续 `align_adjacent_slices_ot` 会对每片做刚性旋转+平移，把不同坐标系叠到同一参考系
    # Z：按切片顺序分层；`z_spacing` 控制层间距（XY 很大时建议调大，否则 3D 观感会很“扁”）
    z_spacing=50.0,
    z_base=0.0,
    # 不信任 sampleid 的顺序：用表达相似自动推断切片前后关系
    order_mode="infer",
    # 推断顺序时使用 OT 距离 + 全局最短路径（强调全局流畅性）
    order_config=scb.OrderConfig(subsample_n=2000, svd_dim=50, knn=30, use_ot=True, ot_reg=0.05),
)

print("n_slices =", len(cube.adatas))
print("first 5 slice infos:")
for info in cube.slice_infos()[:5]:
    print(info)

/cluster2/huanglab/jiamao/conda/envs/spatiocube/lib/python3.10/site-packages/ot/bregman/_sinkhorn.py:630: RuntimeWarning: divide by zero encountered in divide
  v = b / KtransposeU
/cluster2/huanglab/jiamao/conda/envs/spatiocube/lib/python3.10/site-packages/ot/bregman/_sinkhorn.py:642: UserWarning: Warning: numerical errors at iteration 0
  warnings.warn("Warning: numerical errors at iteration %d" % ii)


n_slices = 10
first 5 slice infos:
SliceInfo(key='T309', z=0.0, n_obs=10592, extra=None)
SliceInfo(key='T308', z=50.0, n_obs=10177, extra=None)
SliceInfo(key='T307', z=100.0, n_obs=10088, extra=None)
SliceInfo(key='T306', z=150.0, n_obs=9908, extra=None)
SliceInfo(key='T305', z=200.0, n_obs=10140, extra=None)


In [3]:
# ---- 顺序诊断输出：推断的切片顺序 vs 你认知的真实顺序 ----
# 推断后 cube.adatas 的当前顺序（用于后续对齐）
inferred_order = [a.obs["sampleid"].iloc[0] for a in cube.adatas]
print("Inferred sampleid order:")
print(inferred_order)

# 如果你有真实顺序（例如 list[str]），可在这里粘贴对照：
# true_order = ["T300", "T301", ...]
# print("True order:")
# print(true_order)

# 打印相邻对的 spot 数，辅助判断是否存在明显不匹配的相邻对
print("Adjacent pairs (n_obs):")
for i in range(len(cube.adatas) - 1):
    a_tgt = cube.adatas[i]
    a_src = cube.adatas[i + 1]
    print(
        f"{i}: {a_src.obs['sampleid'].iloc[0]} -> {a_tgt.obs['sampleid'].iloc[0]}  "
        f"(src_n={a_src.n_obs}, tgt_n={a_tgt.n_obs})"
    )

Inferred sampleid order:
['T309', 'T308', 'T307', 'T306', 'T305', 'T304', 'T303', 'T302', 'T301', 'T300']
Adjacent pairs (n_obs):
0: T308 -> T309  (src_n=10177, tgt_n=10592)
1: T307 -> T308  (src_n=10088, tgt_n=10177)
2: T306 -> T307  (src_n=9908, tgt_n=10088)
3: T305 -> T306  (src_n=10140, tgt_n=9908)
4: T304 -> T305  (src_n=9944, tgt_n=10140)
5: T303 -> T304  (src_n=9582, tgt_n=9944)
6: T302 -> T303  (src_n=9507, tgt_n=9582)
7: T301 -> T302  (src_n=9883, tgt_n=9507)
8: T300 -> T301  (src_n=8936, tgt_n=9883)


## 2) 相邻切片 OT 对齐（coarse-to-fine）

说明：
- 默认把 slice `i+1` 逐步对齐到 slice `i` 的坐标系（累积对齐）。
- 为了适配 ~1e5 spots，内部会 **subsample** 进行 OT，然后把刚性变换应用到全量坐标。


In [6]:
from spatiocube.align import align_adjacent_slices_ot

# 关键：用表达 embedding KNN 选候选匹配，避免初始坐标偏差导致“对齐不动”
results = align_adjacent_slices_ot(
    cube,
    subsample_n=2000,
    svd_dim=50,
    expr_knn=50,
    ot_reg=0.05,
    clip_quantile=0.95,
    n_iter=3,
    random_state=0,
)

for r in results[:5]:
    print(r)

print("has map_to_prev:", ["map_to_prev" in a.uns.get("SpatioCube", {}) for a in cube.adatas[:5]])

/cluster2/huanglab/jiamao/Project/SpatioCube/src/spatiocube/align.py:165: RuntimeWarning: Falling back to Sinkhorn OT for this adjacent pair: requested transport='emd' but subsample sizes (2000, 1107) are incompatible with linear assignment or exceed `emd_max_n=2000`.
  warnings.warn(
/cluster2/huanglab/jiamao/Project/SpatioCube/src/spatiocube/align.py:165: RuntimeWarning: Falling back to Sinkhorn OT for this adjacent pair: requested transport='emd' but subsample sizes (2000, 1128) are incompatible with linear assignment or exceed `emd_max_n=2000`.
  warnings.warn(
/cluster2/huanglab/jiamao/Project/SpatioCube/src/spatiocube/align.py:165: RuntimeWarning: Falling back to Sinkhorn OT for this adjacent pair: requested transport='emd' but subsample sizes (2000, 1278) are incompatible with linear assignment or exceed `emd_max_n=2000`.
  warnings.warn(


AlignResult(chamfer_xy=7.395464656744211, n_source=10177, n_target=10592, method='coarse_emd_rigid')
AlignResult(chamfer_xy=13.147935724021519, n_source=10088, n_target=10177, method='coarse_emd_rigid')
AlignResult(chamfer_xy=8.294293546408714, n_source=9908, n_target=10088, method='coarse_sinkhorn_rigid')
AlignResult(chamfer_xy=12.597065587666028, n_source=10140, n_target=9908, method='coarse_sinkhorn_rigid')
AlignResult(chamfer_xy=3.212896680373582, n_source=9944, n_target=10140, method='coarse_emd_rigid')
has map_to_prev: [False, True, True, True, True]


In [7]:
import numpy as np

def bbox(xy):
    return xy[:,0].min(), xy[:,0].max(), xy[:,1].min(), xy[:,1].max()

for i, a in enumerate(cube.adatas[:5]):
    xy = np.asarray(a.obsm[cube.spatial_key])
    print(i, a.obs["sampleid"].iloc[0], "bbox:", bbox(xy))

0 T309 bbox: (552.0, 656.625732140488, 0.0, 126.766001198885)
1 T308 bbox: (551.5886010453667, 664.3143492961955, -5.382866854557221, 119.16422374611196)
2 T307 bbox: (552.8422876890335, 660.4074656349005, 0.927232144323213, 128.61732651514143)
3 T306 bbox: (548.8508132323279, 653.8491108456742, 3.893016156044064, 127.18999765344664)
4 T305 bbox: (555.7063935526625, 658.2295182691036, 4.897160709995703, 130.2973184907837)


## 3) 3D 建图 + Leiden 聚类

- `n_intra`: 每个切片内部 KNN 邻居数
- `n_inter`: 相邻切片之间的跨层邻居数


In [8]:
from spatiocube.graph import build_3d_adjacency, leiden_cluster

A = build_3d_adjacency(cube, n_intra=15, n_inter=5, prefer_mapping=True)
labels = leiden_cluster(A, resolution=1.0, random_state=0)
cube.set_clusters(labels)

print("n_labels =", len(set(labels)))
print("first slice cluster counts:")
print(cube.adatas[0].obs[cube.cluster_key].value_counts().head())

n_labels = 20
first slice cluster counts:
SpatioCube_cluster
4     3006
5     2695
14    1783
16    1304
18     951
Name: count, dtype: int64


## 4) 3D 可视化（Plotly）

把所有切片的 `spatial_3d` 拼起来画点云，颜色按 `SpatioCube_cluster`。


In [9]:
import numpy as np

xy0 = np.asarray(cube.adatas[0].obsm[cube.spatial_key])
xy1 = np.asarray(cube.adatas[1].obsm[cube.spatial_key])
print("slice0 bbox:", xy0.min(0), xy0.max(0))
print("slice1 bbox:", xy1.min(0), xy1.max(0))

slice0 bbox: [552.   0.] [656.62573214 126.7660012 ]
slice1 bbox: [551.58860105  -5.38286685] [664.3143493  119.16422375]


In [10]:
import numpy as np

cube.write_back()  # ensure spatial_3d
xyz = np.vstack([a.obsm[cube.spatial_3d_key] for a in cube.adatas])
c = np.concatenate([a.obs[cube.cluster_key].to_numpy() for a in cube.adatas])

fig = scb.plotly_pointcloud(xyz, color=c, size=2.0, title="SpatioCube 3D clusters")
fig.show()

## 5)（可选）对比学习 embedding（需要 torch）

如果你安装了 `torch`，可以在 3D 图上先学一个对比学习 embedding `X_spatiocube`。
然后在 `X_spatiocube` 上再跑 Leiden/kmeans（此处示例只展示如何生成 embedding）。


In [ ]:
# from spatiocube.contrastive import contrastive_embed_3d, ContrastiveConfig
# Z = contrastive_embed_3d(cube, A, config=ContrastiveConfig(epochs=20, batch_size=4096))
# print("Z shape =", Z.shape)
# print("per-slice obsm key:", cube.adatas[0].obsm["X_spatiocube"].shape)
pass